In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. Load and prep the Titanic data (Same pipeline as Week 2)
df = pd.read_csv('titanic.csv')
df['Age'] = df['Age'].fillna(df['Age'].median())
df = df.drop('Cabin', axis=1)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df = df.drop(['PassengerId', 'Name', 'Ticket'], axis=1)
df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

X = df.drop('Survived', axis=1)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Original Model Evaluation
base_model = LogisticRegression(max_iter=1000)
base_model.fit(X_train, y_train)
base_preds = base_model.predict(X_test)

print("--- ORIGINAL MODEL PERFORMANCE ---")
print(classification_report(y_test, base_preds))

# 3. Hyperparameter Tuning using GridSearchCV
# Tuning 2 hyperparameters: 'C' (regularization strength) and 'solver' (algorithm)
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}

# We use scoring='f1' to optimize for the F1-score rather than just accuracy
grid_search = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5, scoring='f1')
grid_search.fit(X_train, y_train)

tuned_model = grid_search.best_estimator_
tuned_preds = tuned_model.predict(X_test)

print("\n--- TUNED MODEL PERFORMANCE ---")
print(f"Best Hyperparameters Found: {grid_search.best_params_}\n")
print(classification_report(y_test, tuned_preds))

# 4. Before/After Comparison Table
base_report = classification_report(y_test, base_preds, output_dict=True)
tuned_report = classification_report(y_test, tuned_preds, output_dict=True)

comparison_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision (Survived)", "Recall (Survived)", "F1-Score (Survived)"],
    "Original Model": [
        base_report['accuracy'],
        base_report['1']['precision'],
        base_report['1']['recall'],
        base_report['1']['f1-score']
    ],
    "Tuned Model": [
        tuned_report['accuracy'],
        tuned_report['1']['precision'],
        tuned_report['1']['recall'],
        tuned_report['1']['f1-score']
    ]
}).round(4)

print("\n--- BEFORE/AFTER COMPARISON ---")
print(comparison_df.to_string(index=False))

--- ORIGINAL MODEL PERFORMANCE ---
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       105
           1       0.79      0.74      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179


--- TUNED MODEL PERFORMANCE ---
Best Hyperparameters Found: {'C': 1, 'solver': 'liblinear'}

              precision    recall  f1-score   support

           0       0.79      0.85      0.82       105
           1       0.76      0.69      0.72        74

    accuracy                           0.78       179
   macro avg       0.78      0.77      0.77       179
weighted avg       0.78      0.78      0.78       179


--- BEFORE/AFTER COMPARISON ---
              Metric  Original Model  Tuned Model
            Accuracy          0.8101       0.7821
Precision (Survived)          0.7857       0.7612
   Recall (Survived)          0.74

**Why Accuracy Alone Can Be Misleading:**

Imagine a dataset where 99% of credit card transactions are legitimate and only 1% are fraud. A lazy model that simply guesses "legitimate" every single time will achieve 99% accuracy, making it look fantastic on paper. However, it completely fails at its actual job of catching fraud. For imbalanced datasets, accuracy hides our mistakes. Instead, metrics like Precision (how many of our positive guesses were actually correct), Recall (how many of the actual positives we successfully found), and the F1-score (the harmonic balance between the two) give us the true, uninflated picture of how well the model is performing.